### Install openpyxl

In [0]:
%python
pip install openpyxl

### Import the modules that are needed

In [0]:
%python
import re
import pandas as pd
from functools import reduce
from pyspark.sql import functions as F



### Load and clean data

In [0]:
%python
raw = "/Volumes/lender_portfolio/bronze/raw_files"

sources = [
    ("customers.xlsx", ["customers_2024_25", "customers_2026"], "customers"),
    ("applications.csv", None, "applications"),
    ("loans.csv", None, "loans"),
    ("repayments.csv", None, "repayments"),
    ("partners.csv", None, "partners"),
    ("campaigns.csv", None, "campaigns")
]

def snake(col: str) -> str:
    return re.sub(r"[^0-9a-z]+", "_", col.strip().lower()).strip("_")

def load_part(fname, sheet):
    path = f"{raw}/{fname}"
    opts = dict(dtype = str, keep_default_na = False, na_values = [""])
    pdf = pd.read_csv(path, **opts) if sheet is None else pd.read_excel(path, sheet_name = sheet, **opts)
    pdf.columns = [snake(c) for c in pdf.columns]
    pdf = pdf.astype(object).where(pdf.notna(), None)
    schema = ", ".join(f"`{c}` string" for c in pdf.columns)
    return (spark.createDataFrame(pdf, schema = schema)
            .withColumn("_source_file", F.lit(fname))
            .withColumn("_source_sheet", F.lit(sheet).cast("string"))
            .withColumn("_ingestd_at", F.current_timestamp()))

for fname, sheets, table in sources:
    parts = [load_part(fname, s) for s in (sheets or [None])]
    df = reduce(lambda a, b: a.unionByName(b), parts)
    (df.write.format("delta").mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable(f"lender_portfolio.bronze.{table}"))
    print(table, df.count())